In [16]:
import os, re

import yaml
import cv2

In [17]:
CROPS_PATH = os.path.join('.', '.data', 'crops')
RAW_PATH = os.path.join('.', '.data', 'raw')

In [18]:
data_info = yaml.load(open(os.path.join(RAW_PATH, 'raw.yaml')), yaml.SafeLoader)

names: list[str] = data_info['names']
nc: int = data_info['nc']

'# Classes: ' + str(nc), ' - '.join(names)

('# Classes: 10',
 'banded_chlorosis - brown_spot - brownrust - dried_leaves - grassy_shoot - pokkah_boeng - sett_rot - smut - viral_disease - yellow_leaf')

In [19]:
IMGS_PATH = os.path.join(RAW_PATH, 'images')
LABELS_PATH = os.path.join(RAW_PATH, 'labels')

imgs = os.listdir(IMGS_PATH)

for c in names:
  os.makedirs(os.path.join(CROPS_PATH, c), exist_ok=True)

for img_name in imgs:
  m = re.match(r"(.+?)_(\d+)_(\d+)_jpg", img_name)

  if not m:
    pass

  image_class=m.group(1)
  video_id=int(m.group(2))
  frame_id=int(m.group(3))
  path=img_name


  with open(os.path.join(LABELS_PATH, img_name).replace(".jpg", ".txt")) as f:
    lines = [ l.strip().split() for l in f.readlines() ]

  for idx, line in enumerate(lines):
    img_class, x, y, w, h = names[int(line[0])], float(line[1]), float(line[2]), float(line[3]), float(line[4])

    if not img_name.startswith(img_class):
      continue

    img = cv2.imread(os.path.join(IMGS_PATH, img_name))

    if img is None:
      raise ValueError("Imagem não encontrada no caminho especificado.")

    H, W = img.shape[:2]

    x_center = x * W
    y_center = y * H
    w_px = w * W
    h_px = h * H

    x1 = int(x_center - w_px / 2)
    y1 = int(y_center - h_px / 2)
    x2 = int(x_center + w_px / 2)
    y2 = int(y_center + h_px / 2)

    crop = img[y1:y2, x1:x2]

    out_path = os.path.join(CROPS_PATH, img_class, f"{image_class}_{video_id}_{frame_id}_{idx}.jpg")

    cv2.imwrite(out_path, crop)